# Tetranet 10M: Full Training + Analysis
---
**Datasets needed:**
- `tinystories-full` (TinyStoriesV2-GPT4-train.txt + TinyStoriesV2-GPT4-valid.txt)
- `tetranet-code` (train_kaggle.py, model.py, quaternary.py, regularization.py, eval_all.py, tetranet_tokenizer.json)

**Run order:** Cell 0 -> **Runtime -> Restart and run all**

In [ ]:
# Cell 0: Fix P100 incompatibility (run once, then Restart & Run All)
!pip install -q torch==2.5.1 --index-url https://download.pytorch.org/whl/cu124
!pip install -q tokenizers matplotlib pandas seaborn

In [ ]:
# Cell 1: Setup - copy code + data from Kaggle Datasets
import glob, json, math, os, shutil, subprocess, sys, time
import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams.update({
    "figure.dpi": 150, "font.size": 11, "axes.titlesize": 13,
    "axes.labelsize": 11, "legend.fontsize": 9,
    "xtick.labelsize": 9, "ytick.labelsize": 9,
})

os.makedirs("/kaggle/working/tetranet", exist_ok=True)
for f in ["train_kaggle.py", "model.py", "quaternary.py", "regularization.py",
          "eval_all.py", "tetranet_tokenizer.json"]:
    shutil.copy(f"/kaggle/input/tetranet-code/{f}", f"/kaggle/working/tetranet/{f}")
os.chdir("/kaggle/working/tetranet")

os.makedirs("./tinystories", exist_ok=True)
for split in ["train", "valid"]:
    shutil.copy(
        f"/kaggle/input/tinystories-full/TinyStoriesV2-GPT4-{split}.txt",
        f"./tinystories/TinyStories-{split}.txt")

DEVICE = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
VRAM = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
print(f"GPU: {DEVICE}")
print(f"VRAM: {VRAM:.1f} GB")

In [ ]:
# Cell 2: Train original 6 baselines at 10M scale
BASELINES_ORIG = [
    ("full_precision", ""),
    ("bitnet", ""),
    ("uniform_2bit", ""),
    ("fixed_c_025", ""),
    ("fixed_c_05", ""),
    ("learned_c", "--c-lr 0.003 --alpha 2.0 --snap-start 0.4"),
]

SMALL_CONFIG = "--hidden-dim 256 --num-layers 8 --num-heads 8 --ffn-dim 1024"

for name, extra_args in BASELINES_ORIG:
    ckpt = f"./checkpoint_{name}.pt"
    log = f"./log_{name}.csv"
    print(f"\n{'='*60}")
    print(f"Training: {name}")
    print(f"{'='*60}")
    cmd = (
        f"python train_kaggle.py --baseline {name} "
        f"--data-path ./tinystories/TinyStories-train.txt "
        f"--tokenizer-path ./tetranet_tokenizer.json "
        f"--max-stories 100000 {SMALL_CONFIG} "
        f"--batch-size 8 --grad-accum 4 "
        f"--lr 3e-4 --weight-decay 0.1 "
        f"--epochs 1 "
        f"--checkpoint-path {ckpt} --log-path {log} "
        f"--ckpt-interval 5000 --log-interval 50 "
        f"--bf16 {extra_args}"
    )
    t0 = time.time()
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    elapsed = (time.time() - t0) / 60
    print(result.stdout)
    if result.stderr:
        print("STDERR:", result.stderr[:500])
    print(f"  -> {name} finished in {elapsed:.1f} min")

print("\n Original 6 baselines trained.")

In [ ]:
# Cell 2B: Train 2 new experimental baselines
BASELINES_NEW = [
    ("heterogeneous", ""),
    ("learned_c_slow", "--c-lr 0.003 --alpha 0.5 --snap-start 0.0"),
]

for name, extra_args in BASELINES_NEW:
    ckpt = f"./checkpoint_{name}.pt"
    log = f"./log_{name}.csv"
    baseline_arg = name.replace("_slow", "")  # learned_c_slow -> learned_c
    print(f"\n{'='*60}")
    print(f"Training: {name} (--baseline {baseline_arg})")
    print(f"{'='*60}")
    cmd = (
        f"python train_kaggle.py --baseline {baseline_arg} "
        f"--data-path ./tinystories/TinyStories-train.txt "
        f"--tokenizer-path ./tetranet_tokenizer.json "
        f"--max-stories 100000 {SMALL_CONFIG} "
        f"--batch-size 8 --grad-accum 4 "
        f"--lr 3e-4 --weight-decay 0.1 "
        f"--epochs 1 "
        f"--checkpoint-path {ckpt} --log-path {log} "
        f"--ckpt-interval 5000 --log-interval 50 "
        f"--bf16 {extra_args}"
    )
    t0 = time.time()
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    elapsed = (time.time() - t0) / 60
    print(result.stdout)
    if result.stderr:
        print("STDERR:", result.stderr[:500])
    print(f"  -> {name} finished in {elapsed:.1f} min")

print("\n New baselines trained.")

In [ ]:
# Cell 3: Load all logs (graceful — skip missing CSVs)
ALL_NAMES = ["full_precision", "bitnet", "uniform_2bit",
             "fixed_c_025", "fixed_c_05", "learned_c",
             "heterogeneous", "learned_c_slow"]
ALL_COLORS = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2", "#937860",
              "#E5AE38", "#7A68A6"]
ALL_LABELS = ["Full precision", "BitNet b1.58", "Uniform 2-bit",
              "Quaternary c=0.25", "Quaternary c=0.5", "Quaternary learned_c (snap=0.4)",
              "Heterogeneous (blueprint)", "learned_c (snap=0.0, slow)"]

logs = {}
trained_names = []
trained_colors = []
trained_labels = []
for i, name in enumerate(ALL_NAMES):
    log_path = f"./log_{name}.csv"
    if not os.path.exists(log_path):
        print(f"{name:20s}  SKIP — CSV not found")
        continue
    df = pd.read_csv(log_path)
    if len(df) == 0:
        print(f"{name:20s}  SKIP — empty log")
        continue
    if "c_values_json" in df.columns:
        df["c_values"] = df["c_values_json"].apply(
            lambda x: json.loads(x) if isinstance(x, str) else {})
    logs[name] = df
    trained_names.append(name)
    trained_colors.append(ALL_COLORS[i])
    trained_labels.append(ALL_LABELS[i])
    print(f"{name:20s}  {len(df):>4d} rows  "
          f"final loss={df['task_loss'].iloc[-1]:.4f}  "
          f"best loss={df['task_loss'].min():.4f}")

# Filter global lists to only trained baselines
ALL_NAMES = trained_names
ALL_COLORS = trained_colors
ALL_LABELS = trained_labels

In [ ]:
# Cell 4: Fig 1 - Training loss overlay (all 8 baselines)
fig, ax = plt.subplots(figsize=(12, 5))
for i, name in enumerate(ALL_NAMES):
    ax.plot(logs[name]["step"], logs[name]["task_loss"],
            color=ALL_COLORS[i], label=ALL_LABELS[i], linewidth=1.5, alpha=0.85)
ax.set_xlabel("Step")
ax.set_ylabel("Cross-entropy loss")
ax.set_title("Training loss curves - 10M models on TinyStories")
ax.legend(framealpha=0.9, ncol=2, fontsize=8)
ax.set_xlim(left=0)
fig.tight_layout()
fig.savefig("fig1_training_loss.png", dpi=150)
plt.show()

In [ ]:
# Cell 5: Fig 2 - Gradient norm comparison
fig, ax = plt.subplots(figsize=(12, 5))
for i, name in enumerate(ALL_NAMES):
    df = logs[name]
    if "grad_norm" in df.columns:
        ax.plot(df["step"], df["grad_norm"], color=ALL_COLORS[i],
                label=ALL_LABELS[i], linewidth=1, alpha=0.8)
ax.set_xlabel("Step")
ax.set_ylabel("Gradient norm")
ax.set_title("Gradient norm during training")
ax.legend(framealpha=0.9, ncol=2, fontsize=8)
ax.set_yscale("log")
fig.tight_layout()
fig.savefig("fig2_grad_norm.png", dpi=150)
plt.show()

In [ ]:
# Cell 6: Fig 3 - PPL comparison bar chart
from train_kaggle import build_model
from eval_all import eval_ppl, load_tokenizer, tokenize_valid, ValidDataset

SEQ_LEN, BATCH_SIZE, MAX_STORIES = 512, 4, 2500
tokenizer = load_tokenizer("./tetranet_tokenizer.json")
tokens = tokenize_valid("./tinystories/TinyStories-valid.txt", tokenizer, MAX_STORIES)
dataset = ValidDataset(tokens, SEQ_LEN)
loader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
print(f"Validation set: {len(dataset):,} sequences\n")

ppl_results = {}
for name in ALL_NAMES:
    ckpt_path = f"./checkpoint_{name}_final.pt"
    if not os.path.exists(ckpt_path):
        ckpt_path = f"./checkpoint_{name}.pt"
    if not os.path.exists(ckpt_path):
        print(f"SKIP {name}: checkpoint not found"); continue
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    baseline_cls = name.replace("_slow", "")
    model = build_model(baseline=baseline_cls, vocab_size=4096, hidden_dim=256,
                        num_layers=8, num_heads=8, ffn_dim=1024)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    loss, ppl = eval_ppl(model, loader)
    ppl_results[name] = {"loss": loss, "ppl": ppl}
    print(f"{name:20s}  loss={loss:.4f}  ppl={ppl:.2f}")

fig, ax = plt.subplots(figsize=(12, 5))
show_names = [n for n in ALL_NAMES if n in ppl_results]
vals = [ppl_results[n]["ppl"] for n in show_names]
labels_plot = [ALL_LABELS[ALL_NAMES.index(n)] for n in show_names]
colors_plot = [ALL_COLORS[ALL_NAMES.index(n)] for n in show_names]
bars = ax.bar(labels_plot, vals, color=colors_plot, width=0.6, edgecolor="white")
ax.set_ylabel("Perplexity (lower is better)")
ax.set_title("Perplexity comparison - TinyStories validation (10M models)")
ax.tick_params(axis="x", rotation=25)

for bar, val in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(vals)*0.01,
            f"{val:.2f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

if "full_precision" in ppl_results:
    fp_ppl = ppl_results["full_precision"]["ppl"]
    for bar, val in zip(bars, vals):
        delta = val - fp_ppl
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()/2,
                f"{delta:+.2f} vs FP", ha="center", va="center",
                fontsize=7, color="white" if np.mean(bar.get_facecolor()[:3]) < 0.5 else "black",
                fontweight="bold")

fig.tight_layout()
fig.savefig("fig3_ppl_comparison.png", dpi=150)
plt.show()

In [ ]:
# Cell 7: Fig 4 - Structural fingerprinting (from original learned_c run)
if os.path.exists("./checkpoint_learned_c_final.pt") or os.path.exists("./checkpoint_learned_c.pt"):
    ckpt_path_lc = "./checkpoint_learned_c_final.pt"
    if not os.path.exists(ckpt_path_lc):
        ckpt_path_lc = "./checkpoint_learned_c.pt"
    ckpt_lc = torch.load(ckpt_path_lc, map_location="cpu", weights_only=False)
    model_lc = build_model(baseline="learned_c", vocab_size=4096, hidden_dim=256,
                           num_layers=8, num_heads=8, ffn_dim=1024)
    model_lc.load_state_dict(ckpt_lc["model_state_dict"])
    c_vals = model_lc.get_c_values()

    proj_map = {"q_proj": "Q", "k_proj": "K", "v_proj": "V", "o_proj": "O",
                "gate_proj": "Gate", "up_proj": "Up", "down_proj": "Down"}
    by_type = {}
    for key, val in c_vals.items():
        for pat, label in proj_map.items():
            if pat in key:
                by_type.setdefault(label, []).append(val); break

    types = sorted(by_type.keys())
    counts_025 = [sum(1 for v in by_type[t] if abs(v - 0.25) < 0.02) for t in types]
    counts_05  = [sum(1 for v in by_type[t] if abs(v - 0.50) < 0.02) for t in types]

    fig, ax = plt.subplots(figsize=(9, 4.5))
    w = 0.35
    b1 = ax.bar(np.arange(len(types)) - w/2, counts_025, w, label="Snapped to 0.25",
                color="#4C72B0", edgecolor="white")
    b2 = ax.bar(np.arange(len(types)) + w/2, counts_05, w, label="Snapped to 0.50",
                color="#DD8452", edgecolor="white")
    ax.set_ylabel("Number of layers (out of 8)")
    ax.set_title("Structural fingerprinting: per-projection c snapping")
    ax.set_xticks(np.arange(len(types))); ax.set_xticklabels(types)
    ax.legend(framealpha=0.9)
    for bar in b1 + b2:
        h = bar.get_height()
        if h > 0:
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.05, int(h),
                    ha="center", va="bottom", fontweight="bold")
    ax.set_ylim(0, max(max(counts_025), max(counts_05)) + 1.5)
    ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    fig.tight_layout()
    fig.savefig("fig4_fingerprinting.png", dpi=150)
    plt.show()
else:
    print("SKIP fingerprinting: learned_c checkpoint not found")

In [ ]:
# Cell 8: Fig 5 - C-value heatmap (from learned_c)
if 'c_vals' in dir() and c_vals:
    proj_order = ["q_proj", "k_proj", "v_proj", "o_proj",
                  "gate_proj", "up_proj", "down_proj"]
    proj_labels = ["Q", "K", "V", "O", "Gate", "Up", "Down"]
    c_matrix = np.full((8, len(proj_order)), np.nan)
    for key, val in c_vals.items():
        for li in range(8):
            if f"layer.{li}." in key:
                for pi, pat in enumerate(proj_order):
                    if pat in key:
                        c_matrix[li, pi] = val; break
                break
    fig, ax = plt.subplots(figsize=(9, 5))
    cmap = sns.diverging_palette(250, 15, s=75, l=45, n=256, center="light")
    sns.heatmap(c_matrix, ax=ax, annot=True, fmt=".4f", cmap=cmap,
                xticklabels=proj_labels, yticklabels=[f"Layer {i}" for i in range(8)],
                vmin=0.15, vmax=0.60, linewidths=1, linecolor="white",
                cbar_kws={"label": "c value", "shrink": 0.8})
    ax.set_title("Learned c values per layer and projection")
    ax.set_xlabel("Projection type")
    ax.set_ylabel("Layer depth")
    fig.tight_layout()
    fig.savefig("fig5_c_heatmap.png", dpi=150)
    plt.show()
else:
    print("SKIP heatmap: c_vals not available")

In [ ]:
# Cell 9: Fig 6 - Snapping loss landscape (original learned_c vs slow learned_c_slow)
snap_names = [n for n in ["learned_c", "learned_c_slow"] if n in logs]
fig, axes = plt.subplots(1, len(snap_names), figsize=(10 * max(len(snap_names), 1), 4))
if len(snap_names) == 1:
    axes = [axes]
for idx, name in enumerate(snap_names):
    df = logs[name]
    ax1 = axes[idx]
    if "total_loss" not in df.columns:
        ax1.text(0.5, 0.5, f"No snap data for {name}", ha="center", va="center")
        continue
    ax1.plot(df["step"], df["task_loss"], color="#4C72B0", linewidth=2, label="Task loss")
    ax1.plot(df["step"], df["total_loss"], color="#C44E52", linewidth=2, linestyle="--", label="Total loss")
    ax2_ = ax1.twinx()
    ax2_.plot(df["step"], df["lambda"], color="#55A868", linewidth=2, linestyle=":", label="l(t)")
    ax2_.set_ylabel("l(t)", color="#55A868")
    ax2_.tick_params(axis="y", labelcolor="#55A868")
    snap_idx = (df["lambda"] > 0).idxmax() if (df["lambda"] > 0).any() else None
    if snap_idx is not None:
        ax1.axvline(x=df.loc[snap_idx, "step"], color="gray", linestyle="--", alpha=0.6)
    ax1.set_xlabel("Step"); ax1.set_ylabel("Loss")
    ax1.set_title(f"{ALL_LABELS[ALL_NAMES.index(name)]}")
    ax1.legend(framealpha=0.9, fontsize=8)
fig.suptitle("Snapping loss landscape comparison", fontsize=13)
fig.tight_layout()
fig.savefig("fig6_snapping_landscape.png", dpi=150)
plt.show()

In [ ]:
# Cell 10: Fig 7 - Snapping convergence comparison
snap_names = [n for n in ["learned_c", "learned_c_slow"] if n in logs]
if snap_names:
    fig, ax = plt.subplots(figsize=(10, 5))
    styles = ["o", "s"]
    for si, name in enumerate(snap_names):
        df = logs[name]
        if "n_snapped_025" not in df.columns:
            continue
        ax.plot(df["step"], df["n_snapped_025"] + df["n_snapped_05"],
                color=ALL_COLORS[ALL_NAMES.index(name)], linewidth=2,
                marker=styles[si], label=f"{ALL_LABELS[ALL_NAMES.index(name)]} - total snapped")
    ax.set_xlabel("Step"); ax.set_ylabel("Total parameters snapped")
    ax.set_title("Snapping convergence comparison")
    ax.legend(framealpha=0.9)
    ax.set_ylim(0, 60)
    ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    fig.tight_layout()
    fig.savefig("fig7_snapping_convergence.png", dpi=150)
    plt.show()
else:
    print("SKIP snapping convergence: no snap baselines found")

In [ ]:
# Cell 11: Fig 8 - C-value trajectories (learned_c_slow if available)
df_traj = logs.get("learned_c_slow", logs.get("learned_c"))
if df_traj is not None and "c_values" in df_traj.columns:
    proj_short = {"q_proj": "Q", "k_proj": "K", "v_proj": "V", "o_proj": "O",
                  "gate_proj": "G", "up_proj": "U", "down_proj": "D"}
    fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharex=True, sharey=True)
    axes = axes.flatten()
    colors_proj = plt.cm.Set2(np.linspace(0, 1, len(proj_short)))
    for layer_idx in range(8):
        ax = axes[layer_idx]
        for pi, (pat, plabel) in enumerate(proj_short.items()):
            key = f"layer.{layer_idx}.{pat}"
            traj, steps = [], []
            for _, row in df_traj.iterrows():
                cv = row["c_values"].get(key)
                if cv is not None:
                    traj.append(cv); steps.append(row["step"])
            if traj:
                ax.plot(steps, traj, color=colors_proj[pi], linewidth=1.5, label=plabel, alpha=0.8)
        ax.axhline(y=0.25, color="gray", linestyle=":", alpha=0.4)
        ax.axhline(y=0.50, color="gray", linestyle=":", alpha=0.4)
        ax.set_title(f"Layer {layer_idx}")
        if layer_idx >= 4: ax.set_xlabel("Step")
        if layer_idx % 4 == 0: ax.set_ylabel("c value")
        ax.set_ylim(0.15, 0.60)
    axes[-1].legend(loc="center", framealpha=0.9, ncol=1, bbox_to_anchor=(0.5, 0.5))
    fig.suptitle("C-value trajectories per layer during training", fontsize=14)
    fig.tight_layout()
    fig.savefig("fig8_c_trajectories.png", dpi=150)
    plt.show()
else:
    print("SKIP c-value trajectories: no c_values data")

In [ ]:
# Cell 12: Fig 9 - Energy / MAC comparison
# Energy per op (Horowitz ISSCC 2014, 45nm CMOS, pJ)
FP32_MULT = 3.7
FP32_ADD = 0.9
SHIFT = 0.1
MUX = 0.1

QUANT_ENERGY = {
    "full_precision": FP32_MULT + FP32_ADD,  # 4.6 pJ — FP32 MAC
    "uniform_2bit": FP32_MULT + FP32_ADD,    # 4.6 pJ — NOT power-of-two, needs FP32 MULT
    "bitnet": MUX + FP32_ADD,                 # 1.0 pJ — MUX/neg + ADD, no MULT
    "fixed_c_025": SHIFT + MUX + FP32_ADD,    # 1.1 pJ — SHIFT-2 + MUX + ADD
    "fixed_c_05": SHIFT + MUX + FP32_ADD,     # 1.1 pJ — SHIFT-1 + MUX + ADD
    "fixed_c_075": SHIFT + MUX + FP32_ADD,    # 1.1 pJ — SHIFT + MUX + ADD
    "learned_c": SHIFT + MUX + FP32_ADD,      # 1.1 pJ — snaps to power-of-two
    "heterogeneous": SHIFT + MUX + FP32_ADD,  # 1.1 pJ — mix of SHIFT-1/2
    "learned_c_slow": SHIFT + MUX + FP32_ADD, # 1.1 pJ — snaps to power-of-two
}

def count_macs_per_token(cfg):
    h, ffn, T = cfg["hidden_dim"], cfg["ffn_dim"], cfg["max_seq_len"]
    nh, dh = cfg["num_heads"], h // cfg["num_heads"]
    nl, voc = cfg["num_layers"], cfg["vocab_size"]
    qkv = 3 * h * h
    qk = nh * T * dh
    av = nh * T * dh
    o = h * h
    attn = qkv + qk + av + o
    ffn_tot = 3 * h * ffn
    layer = attn + ffn_tot
    all_layers = layer * nl
    lm = h * voc
    total = all_layers + lm
    return {"attention_per_layer": attn, "ffn_per_layer": ffn_tot,
            "per_layer": layer, "all_layers": all_layers,
            "lm_head": lm, "total_macs": total}

cfg_10m = dict(vocab_size=4096, hidden_dim=256, num_layers=8,
               num_heads=8, ffn_dim=1024, max_seq_len=512)
macs = count_macs_per_token(cfg_10m)

energy_baselines = ["full_precision", "uniform_2bit", "bitnet",
                    "fixed_c_025", "fixed_c_05", "heterogeneous"]
energy_labels = ["Full precision\n(FP32 MULT+ADD)", "Uniform 2-bit\n(FP32 MULT+ADD)",
                 "BitNet b1.58\n(MUX/neg + ADD)", "Quaternary c=0.25\n(SHIFT-2 + ADD)",
                 "Quaternary c=0.5\n(SHIFT-1 + ADD)", "Heterogeneous\n(blueprint)"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# MAC distribution pie
per_layer_attn = (
    3 * cfg_10m["hidden_dim"] * cfg_10m["hidden_dim"]
    + cfg_10m["num_heads"] * cfg_10m["max_seq_len"] * (cfg_10m["hidden_dim"] // cfg_10m["num_heads"]) * 2
    + cfg_10m["hidden_dim"] * cfg_10m["hidden_dim"]
)
per_layer_ffn = 3 * cfg_10m["hidden_dim"] * cfg_10m["ffn_dim"]
ax1.pie([per_layer_attn * cfg_10m["num_layers"], per_layer_ffn * cfg_10m["num_layers"]],
        labels=[f"Attention\n({per_layer_attn:,} MACs/layer)",
                f"FFN\n({per_layer_ffn:,} MACs/layer)"],
        autopct="%1.1f%%", colors=["#4C72B0", "#DD8452"],
        startangle=90, textprops={"fontsize": 10})
ax1.set_title("MAC distribution per token")

# Energy bar chart (hardware-theoretical, 45nm)
fallback_colors = {"full_precision": "#4C72B0", "uniform_2bit": "#55A868",
                   "bitnet": "#DD8452", "fixed_c_025": "#C44E52",
                   "fixed_c_05": "#8172B2", "heterogeneous": "#E5AE38"}
bar_colors = []
for b in energy_baselines:
    if b in ALL_NAMES:
        bar_colors.append(ALL_COLORS[ALL_NAMES.index(b)])
    else:
        bar_colors.append(fallback_colors.get(b, "#999999"))

pj_per_mac = [QUANT_ENERGY[b] for b in energy_baselines]
nJ_per_tok = [macs["total_macs"] * pj / 1000 for pj in pj_per_mac]
bars = ax2.bar(energy_labels, nJ_per_tok,
               color=bar_colors, width=0.5, edgecolor="white")
ax2.set_ylabel("Energy per token (nJ)")
ax2.set_title("Energy per token (Horowitz ISSCC 2014, 45nm)")
ax2.tick_params(axis="x", rotation=15)
for bar, val in zip(bars, nJ_per_tok):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f"{val:.1f}", ha="center", va="bottom", fontweight="bold")
fp_nJ = nJ_per_tok[0]
for bar, val in zip(bars, nJ_per_tok):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 0.6,
             f"{val/fp_nJ*100:.0f}% of FP", ha="center", va="center",
             fontsize=8, color="white", fontweight="bold")
fig.tight_layout()
fig.savefig("fig9_energy_comparison.png", dpi=150)
plt.show()

In [ ]:
# Cell 12B: Inference throughput + peak VRAM
MODEL_CFG = dict(vocab_size=4096, hidden_dim=256, num_layers=8,
                 num_heads=8, ffn_dim=1024, max_seq_len=512)

throughput = {}
vram_usage = {}
ckpt_sizes = {}

for name in ALL_NAMES:
    ckpt_path = f"./checkpoint_{name}_final.pt"
    if not os.path.exists(ckpt_path):
        ckpt_path = f"./checkpoint_{name}.pt"
    if not os.path.exists(ckpt_path):
        ckpt_sizes[name] = 0
        print(f"{name:20s}  SKIP — no checkpoint")
        continue
    ckpt_sizes[name] = os.path.getsize(ckpt_path)

    baseline_cls = name.replace("_slow", "")
    model = build_model(baseline=baseline_cls, **MODEL_CFG)
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(DEVICE)
    model.eval()

    torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    total_toks = 0
    with torch.no_grad():
        for input_ids, labels in loader:
            input_ids = input_ids.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            model(input_ids, labels=labels)
            total_toks += input_ids.numel()
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0

    peak_mb = torch.cuda.max_memory_allocated() / 1e6
    tok_s = total_toks / elapsed

    throughput[name] = tok_s
    vram_usage[name] = peak_mb
    print(f"{name:20s}  {tok_s:>10,.0f} tok/s  {peak_mb:>8.1f} MB VRAM  "
          f"ckpt={ckpt_sizes[name]/1e6:.2f} MB")

# Plot grouped bar chart
fig, ax1 = plt.subplots(figsize=(12, 5))
x = np.arange(len(ALL_NAMES))
w = 0.35

b1 = ax1.bar(x - w/2, [throughput.get(n, 0) for n in ALL_NAMES], w,
             color=ALL_COLORS, edgecolor="white", label="Throughput")
ax1.set_ylabel("Tokens / second")
ax1.set_title("Actual inference throughput & peak VRAM")

ax2_ = ax1.twinx()
b2 = ax2_.bar(x + w/2, [vram_usage.get(n, 0) for n in ALL_NAMES], w,
              color=["#999999"] * len(ALL_NAMES), edgecolor="white",
              label="Peak VRAM", alpha=0.6)
ax2_.set_ylabel("Peak VRAM (MB)")

ax1.set_xticks(x)
ax1.set_xticklabels(ALL_LABELS, rotation=25, ha="right")
fig.legend(loc="upper right", framealpha=0.9)
fig.tight_layout()
fig.savefig("fig10_throughput_vram.png", dpi=150)
plt.show()

# Print checkpoint sizes
print("\n--- Checkpoint sizes ---")
for name in ALL_NAMES:
    sz = ckpt_sizes.get(name, 0)
    print(f"  {name:20s}  {sz/1e6:.2f} MB")


In [ ]:
# Cell 12C: Quantized weight level distribution
def get_weight_distribution(weight, c, threshold=1.0):
    gamma = weight.abs().mean().clamp(min=1e-8).detach()
    scaled = weight / gamma
    boundary = (1.0 + c) / 2.0
    n_neg1 = (scaled <= -boundary).sum().item()
    n_negc = ((scaled > -boundary) & (scaled <= 0)).sum().item()
    n_posc = ((scaled > 0) & (scaled <= boundary)).sum().item()
    n_pos1 = (scaled > boundary).sum().item()
    total = n_neg1 + n_negc + n_posc + n_pos1
    if total == 0: return {}
    neg_c_label = f"-{c:.2f}"
    pos_c_label = f"{c:.2f}"
    return {"-1": n_neg1/total*100, neg_c_label: n_negc/total*100,
            pos_c_label: n_posc/total*100, "+1": n_pos1/total*100}

distributions_raw = {}
for name in ALL_NAMES:
    ckpt_path = f"./checkpoint_{name}_final.pt"
    if not os.path.exists(ckpt_path):
        ckpt_path = f"./checkpoint_{name}.pt"
    if not os.path.exists(ckpt_path):
        continue
    baseline_cls = name.replace("_slow", "")
    model = build_model(baseline=baseline_cls, **MODEL_CFG)
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    first_lin = model.transformer.h[0].attention.q_proj
    w = first_lin.weight.detach()
    c_val = first_lin.c.item() if hasattr(first_lin, "c") else 0.375
    dist = get_weight_distribution(w, c_val)
    distributions_raw[name] = (dist, c_val)

    neg_c_pct = sum(v for k, v in dist.items() if k.startswith("-") and k != "-1") or 0
    pos_c_pct = sum(v for k, v in dist.items() if k != "-1" and k != "+1" and not k.startswith("-")) or 0
    print(f"{name:20s}  c={c_val:.4f}  "
          f"-1: {dist['-1']:.1f}%  -c: {neg_c_pct:.1f}%  "
          f"+c: {pos_c_pct:.1f}%  +1: {dist['+1']:.1f}%")

# Stacked horizontal bar chart
dist_summary = {}
for name in ALL_NAMES:
    if name not in distributions_raw:
        continue
    dist, c_val = distributions_raw[name]
    neg_c = sum(v for k, v in dist.items() if k.startswith("-") and k != "-1")
    pos_c = sum(v for k, v in dist.items() if k != "-1" and k != "+1" and not k.startswith("-"))
    dist_summary[name] = {"-1": dist["-1"], "-c": neg_c, "+c": pos_c, "+1": dist["+1"]}

fig, ax = plt.subplots(figsize=(10, 5))
level_names = ["-1", "-c", "+c", "+1"]
level_colors = ["#C44E52", "#DD8452", "#55A868", "#4C72B0"]
bottom = np.zeros(len(ALL_NAMES))

for li, lname in enumerate(level_names):
    vals = [dist_summary.get(n, {}).get(lname, 0) for n in ALL_NAMES]
    ax.barh(ALL_LABELS, vals, left=bottom.copy(), color=level_colors[li],
            label=lname, height=0.6, edgecolor="white")
    bottom += vals

ax.set_xlabel("Weight proportion (%)")
ax.set_title("Quantized weight distribution per baseline (Q projection, layer 0)")
ax.legend(loc="lower right", framealpha=0.9)
ax.set_xlim(0, 105)
fig.tight_layout()
fig.savefig("fig11_weight_distribution.png", dpi=150)
plt.show()


In [ ]:
# Cell 13: Summary table
print("\n" + "=" * 125)
print("FINAL SUMMARY: Tetranet 10M Model Comparison")
print("=" * 125)
print(f"\nSystem: {DEVICE} ({VRAM:.1f} GB) | "
      f"Model: 10M params (dim=256, layers=8, heads=8, ffn=1024)")
print(f"Dataset: 100K TinyStories (~3.3M tokens) | "
      f"Validation: 2,500 TinyStories (~80K tokens)")

header = (f"\n{'Baseline':<25s} {'Best loss':>10s} {'PPL':>8s} {'DFP':>8s} "
          f"{'tok/s':>10s} {'MB VRAM':>9s} {'Ckpt MB':>8s} "
          f"{'nJ/tok':>8s} {'vs FP':>8s}")
print(header); print("-" * 110)

for name in ALL_NAMES:
    if name not in logs: continue
    best_loss = logs[name]["task_loss"].min()
    ppl = math.exp(best_loss) if best_loss < 10 else float("nan")
    delta_ppl = ppl - ppl_results.get("full_precision", {}).get("ppl", 0)
    qb_map = {"full_precision": "full_precision", "bitnet": "bitnet",
              "uniform_2bit": "full_precision", "fixed_c_025": "fixed_c_025",
              "fixed_c_05": "fixed_c_05", "learned_c": "fixed_c_05",
              "heterogeneous": "fixed_c_05", "learned_c_slow": "fixed_c_05"}
    qb = qb_map.get(name, "full_precision")
    nJ = macs["total_macs"] * QUANT_ENERGY.get(qb, 4.6) / 1000
    fp_nj = macs["total_macs"] * QUANT_ENERGY["full_precision"] / 1000
    label = ALL_LABELS[ALL_NAMES.index(name)]
    tok_s = f"{throughput.get(name, 0):>9,.0f}" if throughput.get(name, 0) else "   N/A"
    vram_str = f"{vram_usage.get(name, 0):>8.0f}" if vram_usage.get(name, 0) else "     N/A"
    ckpt_str = f"{ckpt_sizes.get(name, 0)/1e6:>7.2f}" if ckpt_sizes.get(name, 0) else "   N/A"
    print(f"{label:<25s} {best_loss:>10.4f} {ppl:>8.2f} {delta_ppl:>+8.2f} "
          f"{tok_s:>10s} {vram_str:>9s} {ckpt_str:>8s} "
          f"{nJ:>8.1f} {nJ/fp_nj*100:>7.0f}%")

print("\n--- Snapping statistics ---")
for snap_name in ["learned_c", "learned_c_slow"]:
    if snap_name not in logs or "n_snapped_025" not in logs[snap_name].columns:
        continue
    df_snap = logs[snap_name]
    last_025 = df_snap["n_snapped_025"].iloc[-1]
    last_05 = df_snap["n_snapped_05"].iloc[-1]
    label = ALL_LABELS[ALL_NAMES.index(snap_name)] if snap_name in ALL_NAMES else snap_name
    print(f"  {label:30s}  0.25={last_025}/56  0.5={last_05}/56  total={last_025+last_05}/56")

print("=" * 125)

In [ ]:
# Cell 14: Copy all outputs
for fname in (glob.glob("*.png") + glob.glob("*_final.pt") + glob.glob("*.csv")):
    try: shutil.copy(fname, "/kaggle/working/")
    except: pass
print("\nFinal outputs:")
for f in sorted(os.listdir("/kaggle/working/")):
    sz = os.path.getsize(f"/kaggle/working/{f}")
    print(f"  {f:40s} {sz/1024:>8.1f} KB")
print("\nDone!")